# SARIMAX — teste final walk-forward

Parâmetros congelados, mesmas origens dos quatro modelos, horizonte de 1 hora e reajuste em **toda origem**. Esta é uma execução pesada.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
params = load_json(require_file(
    config.RESULT_DIR / "best_params_SARIMAX.json",
    "01_otimizacao_hiperparametros.ipynb",
))
origins = models.final_origins(model_df)
print("Parâmetros:", params)
print("Origens finais:", len(origins))
res = models.evaluate(model_df, origins, "SARIMAX", params)

models.save_result(res, config.RESULT_DIR)

# Diagnóstico residual em 168 horas consecutivas. Essas previsões não alteram
# o MAE oficial nem a seleção; servem para ACF e Ljung-Box com espaçamento 1h.
diagnostic_origins = models.residual_diagnostic_origins(model_df)
diagnostic = models.evaluate(
    model_df, diagnostic_origins, "SARIMAX", params, progress_every=24
)
diagnostic.predictions.to_csv(
    config.RESULT_DIR / "residual_diagnostic_SARIMAX.csv", index=False
)

baseline_path = config.RESULT_DIR / "baseline_predictions.csv"
baseline = models.baseline_predictions(model_df, origins)
baseline.to_csv(baseline_path, index=False)

print(f"MAE final = {res.mae:.3f}")
print(f"Tempo = {res.runtime_sec / 60:.1f} min")
print(f"Diagnóstico residual consecutivo = {diagnostic.runtime_sec / 60:.1f} min")
display(res.predictions.head())
display(res.predictions.tail())


Parâmetros: {'order': [2, 0, 1], 'seasonal_order': [0, 1, 1, 24]}
Origens finais: 521


C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 10/521 origens | MAE parcial=249.328
SARIMAX: 20/521 origens | MAE parcial=276.271
SARIMAX: 30/521 origens | MAE parcial=276.302
SARIMAX: 40/521 origens | MAE parcial=263.679
SARIMAX: 50/521 origens | MAE parcial=227.972
SARIMAX: 60/521 origens | MAE parcial=223.077
SARIMAX: 70/521 origens | MAE parcial=246.233
SARIMAX: 80/521 origens | MAE parcial=265.447
SARIMAX: 90/521 origens | MAE parcial=288.861
SARIMAX: 100/521 origens | MAE parcial=276.167
SARIMAX: 110/521 origens | MAE parcial=281.961
SARIMAX: 120/521 origens | MAE parcial=288.670
SARIMAX: 130/521 origens | MAE parcial=289.609
SARIMAX: 140/521 origens | MAE parcial=289.078
SARIMAX: 150/521 origens | MAE parcial=284.487
SARIMAX: 160/521 origens | MAE parcial=294.838
SARIMAX: 170/521 origens | MAE parcial=290.872
SARIMAX: 180/521 origens | MAE parcial=291.014
SARIMAX: 190/521 origens | MAE parcial=290.506
SARIMAX: 200/521 origens | MAE parcial=286.381


C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 210/521 origens | MAE parcial=287.196


...

SARIMAX: 144/168 origens | MAE parcial=272.946


C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 168/168 origens | MAE parcial=302.307
MAE final = 263.678
Tempo = 465.7 min
Diagnóstico residual consecutivo = 187.4 min


,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
0,22908,2018-02-04 07:00:00,2018-02-04 08:00:00,1,1857.0,1535.455558,321.544442,46.324645,True,False,True
1,22919,2018-02-04 18:00:00,2018-02-04 19:00:00,1,1720.0,1483.286805,236.713195,45.512910,True,False,True
2,22930,2018-02-05 05:00:00,2018-02-05 06:00:00,1,4846.0,4360.315285,485.684715,35.569847,True,False,True
3,22941,2018-02-05 16:00:00,2018-02-05 17:00:00,1,5912.0,5622.741112,289.258888,54.231025,True,False,False
4,22952,2018-02-06 03:00:00,2018-02-06 04:00:00,1,850.0,714.300626,135.699374,33.794803,True,False,True


,origin_index,forecast_origin,target_date,horizon_hours,y_true,y_pred,residual,elapsed_seconds,target_observed,fallback_used,converged
516,28584,2018-09-28 19:00:00,2018-09-28 20:00:00,1,3538.0,3711.419891,-173.419891,32.232341,True,False,True
517,28595,2018-09-29 06:00:00,2018-09-29 07:00:00,1,2036.0,2005.287276,30.712724,52.357666,True,False,True
518,28606,2018-09-29 17:00:00,2018-09-29 18:00:00,1,4569.0,4172.877308,396.122692,49.407073,True,False,True
519,28617,2018-09-30 04:00:00,2018-09-30 05:00:00,1,444.0,1616.055817,-1172.055817,79.303314,True,False,True
520,28628,2018-09-30 15:00:00,2018-09-30 16:00:00,1,4283.0,4556.595894,-273.595894,39.443105,True,False,True


O arquivo de métricas registra o número de previsões e o tempo médio por origem. Não compare o MAE bruto com bases de outra escala.